# TEP Fault Classification — LightGBM

Multiclass classifier (22 classes: 0 = normal, IDV 1–21 = faults) trained on post-injection process measurements.

In [ ]:
import gc
import pandas as pd
import numpy as np
import lightgbm as lgb
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score
)

sns.set_theme(style="whitegrid", palette="tab10")
plt.rcParams["figure.dpi"] = 120

DATA_DIR = "../datasets/TEP/Harvard"
FF_TRAIN = f"{DATA_DIR}/ff_train.parquet"
F_TRAIN  = f"{DATA_DIR}/f_train.parquet"
FF_TEST  = f"{DATA_DIR}/ff_test.parquet"
F_TEST   = f"{DATA_DIR}/f_test.parquet"

XMEAS    = [f"xmeas_{i}" for i in range(1, 42)]
XMV      = [f"xmv_{i}"   for i in range(1, 12)]
FEATURES = XMEAS + XMV

FAULT_INJECTION_SAMPLE = 160

print("Libraries loaded.")

## 1. Build Training Set

Normal samples from `ff_train` + post-injection faulty samples from `f_train` (fault-by-fault to keep RAM low).

In [ ]:
# Normal samples (label = 0)
print("Loading normal training samples...")
ff = pd.read_parquet(FF_TRAIN, columns=FEATURES)
ff["label"] = 0

# Faulty samples: load one fault at a time, post-injection only
print("Loading faulty training samples (fault-by-fault)...")
chunks = [ff]
for fnum in range(1, 22):
    chunk = pd.read_parquet(
        F_TRAIN,
        columns=["faultNumber", "sample"] + FEATURES,
        filters=[("faultNumber", "==", fnum), ("sample", ">", FAULT_INJECTION_SAMPLE)],
    )
    chunk = chunk[(chunk["faultNumber"] == fnum) & (chunk["sample"] > FAULT_INJECTION_SAMPLE)]
    chunk = chunk[FEATURES].copy()
    chunk["label"] = fnum
    chunks.append(chunk)
    del chunk

train = pd.concat(chunks, ignore_index=True)
del chunks
gc.collect()

X_train = train[FEATURES].values
y_train = train["label"].values.astype(np.int32)
del train
gc.collect()

print(f"Train: {X_train.shape[0]:,} samples, {X_train.shape[1]} features")
print(f"Classes: {np.unique(y_train)}")

## 2. Build Test Set

In [ ]:
# Normal test samples
print("Loading normal test samples...")
ff_te = pd.read_parquet(FF_TEST, columns=FEATURES)
ff_te["label"] = 0

# Faulty test samples — all faults, post-injection only, single read
print("Loading faulty test samples...")
f_te = pd.read_parquet(
    F_TEST,
    columns=["faultNumber", "sample"] + FEATURES,
    filters=[("sample", ">", FAULT_INJECTION_SAMPLE)],
)
f_te = f_te[f_te["sample"] > FAULT_INJECTION_SAMPLE].copy()
f_te["label"] = f_te["faultNumber"].astype(np.int32)
f_te = f_te[FEATURES + ["label"]]

test = pd.concat([ff_te, f_te], ignore_index=True)
del ff_te, f_te
gc.collect()

X_test = test[FEATURES].values
y_test = test["label"].values.astype(np.int32)
del test
gc.collect()

print(f"Test:  {X_test.shape[0]:,} samples, {X_test.shape[1]} features")

## 3. Train LightGBM

In [ ]:
dtrain = lgb.Dataset(X_train, label=y_train, feature_name=FEATURES, free_raw_data=True)
dvalid = lgb.Dataset(X_test,  label=y_test,  feature_name=FEATURES, free_raw_data=True, reference=dtrain)

params = {
    "objective":        "multiclass",
    "num_class":        22,
    "metric":           "multi_logloss",
    "learning_rate":    0.1,
    "num_leaves":       127,
    "min_child_samples": 50,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq":     5,
    "verbose":          -1,
}

callbacks = [
    lgb.early_stopping(stopping_rounds=20, verbose=True),
    lgb.log_evaluation(period=20),
]

model = lgb.train(
    params,
    dtrain,
    num_boost_round=300,
    valid_sets=[dvalid],
    callbacks=callbacks,
)

print(f"\nBest iteration: {model.best_iteration}")

## 4. Evaluate

In [ ]:
y_pred = model.predict(X_test).argmax(axis=1)

print(f"Overall accuracy: {accuracy_score(y_test, y_pred):.4f}")
print()
labels = ["Normal"] + [f"IDV {i}" for i in range(1, 22)]
label_ids = list(range(22))
print(classification_report(y_test, y_pred, labels=label_ids, target_names=labels))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred, labels=label_ids, normalize="true")

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(
    cm, ax=ax, cmap="Blues", vmin=0, vmax=1,
    xticklabels=labels, yticklabels=labels,
    linewidths=0.3, annot=False,
    cbar_kws={"label": "Recall"},
)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Confusion Matrix (row-normalised)")
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Feature importances
imp = pd.Series(model.feature_importance(importance_type="gain"), index=FEATURES)
imp = imp.sort_values(ascending=True).tail(30)

fig, ax = plt.subplots(figsize=(10, 8))
imp.plot(kind="barh", ax=ax, color="steelblue")
ax.set_title("Top 30 Feature Importances (gain)")
ax.set_xlabel("Gain")
plt.tight_layout()
plt.show()